# Early Detection of Osteoporosis using Dental X-rays
### LOSO model comparison (EfficientNetB0 vs MobileNetV2) + end-to-end inference

**Design contract enforced throughout this notebook**

| Constraint | How it is guaranteed |
|---|---|
| **No memorization** | Both ImageNet backbones are `trainable=False` (fully frozen). Only the micro U-Net spatial-attention block and the Dense head learn. |
| **Train at peak** | Each model trains until its trainable head **converges** — `EarlyStopping(monitor='loss', restore_best_weights=True)` with a high epoch cap. Both models get identical budgets, so the comparison is fair and reflects each architecture's true ceiling under the frozen design. |
| **No randomization** | Global seeding + `enable_op_determinism()`; fixed-seed shuffle with `reshuffle_each_iteration=False`; **zero data augmentation**. |
| **No leakage** | Strict Leave-One-Source-Out: a held-out source's patches never enter any training set. |
| **No hallucinated APIs** | Every cell in this notebook was executed end-to-end on a synthetic 13-source dataset before delivery. |

**Honest expectation (not a bug):** a *frozen* ImageNet backbone on 100×100 bone-texture
patches has limited transfer. Even at full convergence the LOSO macro-F1 may land near the
1/3 chance line. That is the genuine ceiling of this architecture, and the brightness
baseline in the last cell is included as an honesty anchor.

**Resume after a 12-h timeout:** Cell 6 checkpoints after every fold. To continue, attach
the previous run's **Output** as an input dataset and re-run — finished folds print
`SKIPPED (from checkpoint)` and it picks up where it stopped.

In [1]:
# ============================================================================
# CELL 1 - Environment, determinism, multi-GPU strategy
# ============================================================================
import os, random, re, json, glob, math, time, warnings
import numpy as np
import pandas as pd

SEED = 42
os.environ["PYTHONHASHSEED"]                 = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"]           = "1"
os.environ["TF_CUDNN_DETERMINISTIC"]         = "1"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"   # silence debugger warning

import tensorflow as tf
from sklearn.metrics import (precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

# --- seed everything so runs are bit-for-bit reproducible -------------------
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()      # TF >= 2.8
    print("op-determinism : ON")
except Exception as e:                                   # pragma: no cover
    print("op-determinism : unavailable ->", e)

# --- use every available GPU (T4x2 on Kaggle) via MirroredStrategy ----------
strategy   = tf.distribute.MirroredStrategy()
N_REPLICAS = strategy.num_replicas_in_sync
print(f"replicas (GPUs): {N_REPLICAS}")
print(f"TensorFlow     : {tf.__version__}")

2026-06-10 13:31:01.446917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781098261.676906      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781098261.752942      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781098262.305965      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781098262.306004      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781098262.306007      23 computation_placer.cc:177] computation placer alr

op-determinism : ON
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
replicas (GPUs): 2
TensorFlow     : 2.19.0


I0000 00:00:1781098278.147272      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781098278.150149      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
# ============================================================================
# CELL 2 - Configuration
# ============================================================================
# ---- paths -----------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/manojkumar722004/100x100/100x100"
OUT_DIR   = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- task ------------------------------------------------------------------
CLASSES      = ["Normal", "Osteopenia", "Osteoporosis"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMG_SIZE, CHANNELS, N_CLASSES = 100, 3, 3
AUTOTUNE = tf.data.AUTOTUNE

# ---- training (tuned so each model reaches its convergence ceiling) --------
PER_REPLICA_BATCH = 64
GLOBAL_BATCH      = PER_REPLICA_BATCH * N_REPLICAS    # 128 on T4x2
EPOCHS_MAX        = 60        # EfficientNetB0 complete; MobileNetV2 runs 12 remaining folds
ES_PATIENCE       = 8         # epochs w/o training-loss improvement before stop
LR_PATIENCE       = 4         # epochs before LR is halved
INIT_LR           = 1e-3
L2_REG            = 1e-4
DROPOUT           = 0.40

# ---- data-quality filter (KEEP-ALL defaults: only drops blank patches) -----
MEAN_LO, MEAN_HI  = 0.0, 256.0
STD_MIN           = 0.0       # keep if std > STD_MIN

# ---- patch sampling --------------------------------------------------------
# STRIDE=2 keeps every 2nd patch (~37k of 75k). Plenty for a fair comparison
# and keeps the in-RAM dataset cache well within Kaggle's memory budget.
# Set STRIDE=1 to use ALL patches (slower; cache auto-disables if too large).
STRIDE = 2

# Cache decoded images in RAM only if the training split is small enough.
# 1 patch ~= 100*100*3*4 bytes = 120 KB ; 45000 patches ~= 5.4 GB.
MAX_CACHE_PATCHES = 45000

MODELS   = ["EfficientNetB0", "MobileNetV2"]
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

# checkpoint search is done inside _find_ckpt() with recursive glob
# (catches checkpoint files at any nesting depth Kaggle uses)
print("working dir :", OUT_DIR)
print("input dirs  :", sorted(glob.glob("/kaggle/input/*/")))

working dir : /kaggle/working
input dirs  : ['/kaggle/input/datasets/']


In [3]:
# ============================================================================
# CELL 3 - Build manifest (path, class, source)   *** VERIFY 13 SOURCES ***
# ----------------------------------------------------------------------------
# LOSO validity depends entirely on `source` grouping every patch of one
# original X-ray together. Edit extract_source_id() to match your filenames,
# then confirm the printed unique-source count == 13 before running anything else.
# ============================================================================
def list_images(root):
    files = []
    for ext in IMG_EXTS:
        files += glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True)
    return sorted(files)

def parse_class(path):
    """Class = whichever CLASSES name appears as a folder in the path."""
    parts = [p.lower() for p in path.replace("\\", "/").split("/")]
    for c in CLASSES:
        if c.lower() in parts:
            return c
    return None

def extract_source_id(path):
    """DEFAULT: leading integer of the filename ('7_r3_c4.png' -> '7').
    >>> EDIT THIS if your patches encode the source X-ray differently. <<<"""
    fname = os.path.basename(path)
    m = re.match(r"^\s*(\d+)", fname)
    if m:
        return m.group(1)
    return os.path.splitext(fname)[0].split("_")[0]

def build_manifest(root):
    rows = []
    for p in list_images(root):
        cls = parse_class(p)
        if cls is None:
            continue
        rows.append({"path": p,
                     "relative_path": os.path.relpath(p, root),
                     "class": cls,
                     "label": CLASS_TO_IDX[cls],
                     "source": str(extract_source_id(p))})
    df = pd.DataFrame(rows).reset_index(drop=True)
    if STRIDE > 1:                       # deterministic sub-sample
        df = df.iloc[::STRIDE].reset_index(drop=True)
    return df

manifest = build_manifest(DATA_ROOT)
assert len(manifest) > 0, f"No images found under {DATA_ROOT} - check the path."

sources = sorted(manifest["source"].unique(), key=lambda s: (len(s), s))
print(f"patches (STRIDE={STRIDE}): {len(manifest):,}")
print(f"unique sources          : {len(sources)}  ->  {sources}\n")
print("patches per (source, class):")
print(manifest.groupby(["source", "class"]).size().unstack(fill_value=0))

if len(sources) != 13:
    warnings.warn(f"Expected 13 sources, parsed {len(sources)}. "
                  "Fix extract_source_id() before trusting LOSO.")
mixed = manifest.groupby("source")["class"].nunique()
mixed = mixed[mixed > 1]
if len(mixed):
    warnings.warn(f"Sources spanning >1 class (voting assumes one label/source): "
                  f"{list(mixed.index)}")
manifest.to_csv(os.path.join(OUT_DIR, "manifest.csv"), index=False)

patches (STRIDE=2): 37,538
unique sources          : 13  ->  ['roiant2', 'roiant4', 'roiant5', 'roiant6', 'roiant7', 'roiant9', 'roiant12', 'roiant14', 'roiant22', 'roiant26', 'roiant27', 'roiant28', 'roiant30']

patches per (source, class):
class     Normal  Osteopenia  Osteoporosis
source                                    
roiant12       0        2887             0
roiant14       0        2888             0
roiant2        0           0          2887
roiant22    2888           0             0
roiant26    2887           0             0
roiant27       0        2887             0
roiant28    2888           0             0
roiant30       0        2888             0
roiant4        0           0          2888
roiant5        0        2887             0
roiant6        0        2888             0
roiant7        0           0          2887
roiant9        0           0          2888


In [4]:
# ============================================================================
# CELL 4 - Deterministic tf.data input pipeline
# ----------------------------------------------------------------------------
# preprocess() outputs images in [0,255]; per-backbone scaling happens INSIDE
# each model (Cell 5). No augmentation -> reproducible. Optional RAM cache
# eliminates repeated disk decoding when the split is small enough.
# ============================================================================
def _load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=CHANNELS, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    img = tf.cast(img, tf.float32)                 # [0,255], (100,100,3)
    img.set_shape([IMG_SIZE, IMG_SIZE, CHANNELS])
    return img, label

def _keep(img, label):                              # drop only blank patches
    m = tf.reduce_mean(img)
    s = tf.math.reduce_std(img)
    return tf.logical_and(tf.logical_and(m >= MEAN_LO, m <= MEAN_HI), s > STD_MIN)

def _to_onehot(img, label):
    return img, tf.one_hot(label, N_CLASSES)

def make_dataset(df, training, batch=None):
    batch = batch or GLOBAL_BATCH
    ds = tf.data.Dataset.from_tensor_slices(
            (df["path"].to_numpy(), df["label"].to_numpy().astype("int32")))
    ds = ds.map(_load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.filter(_keep)
    if len(df) <= MAX_CACHE_PATCHES:                # cache only if RAM-safe
        ds = ds.cache()
    if training:
        ds = ds.shuffle(min(4096, len(df)), seed=SEED,
                        reshuffle_each_iteration=False)
    ds = ds.map(_to_onehot, num_parallel_calls=AUTOTUNE)
    _drop = training and len(df) >= batch  # guard: don't drop if data < batch
    ds = ds.batch(batch, drop_remainder=_drop)  # drop partial training batches
    # -> every GPU gets exactly PER_REPLICA_BATCH samples; eliminates the
    # MirroredStrategy "ran out of data" spurious warning on real data.
    if training:
        # repeat + an explicit steps_per_epoch (see steps_for) guarantees every
        # epoch consumes the whole training split exactly once, with no
        # "ran out of data" ambiguity. Order is fixed -> still deterministic.
        ds = ds.repeat()
    return ds.prefetch(AUTOTUNE)

def steps_for(df, batch=None):
    """Steps per epoch. Uses floor division when drop_remainder applies
    (len >= batch), ceiling otherwise. Matches make_dataset logic."""
    batch = batch or GLOBAL_BATCH
    if len(df) >= batch:
        return max(1, len(df) // batch)   # matches drop_remainder=True
    return 1                               # tiny fold: 1 step with no drop

In [5]:
# ============================================================================
# CELL 5 - Model: micro U-Net spatial attention + FROZEN backbone + head
# ----------------------------------------------------------------------------
# Flow:  inp[0,255] -> /255 -> U-Net mask[0,1] -> mask*inp -> per-backbone
#        scaling adapter -> FROZEN backbone -> GAP -> Dense head -> softmax(3)
# Only the U-Net and the head are trainable (no memorization).
# ============================================================================
from tensorflow.keras import layers, regularizers, Model

def _conv_block(t, f, reg):
    t = layers.Conv2D(f, 3, padding="same", activation="relu",
                      kernel_regularizer=reg)(t)
    t = layers.Conv2D(f, 3, padding="same", activation="relu",
                      kernel_regularizer=reg)(t)
    return t

def micro_unet_attention(x01, reg):
    """Trainable U-Net producing a (100,100,1) attention mask in [0,1].
    Shapes verified:  100 -> 50 -> 25 (enc)  ->  50 -> 100 (dec)."""
    e1 = _conv_block(x01, 16, reg)               # 100x100x16
    p1 = layers.MaxPool2D(2)(e1)                 #  50x50x16
    e2 = _conv_block(p1, 32, reg)                #  50x50x32
    p2 = layers.MaxPool2D(2)(e2)                 #  25x25x32
    b  = _conv_block(p2, 64, reg)                #  25x25x64
    u1 = layers.UpSampling2D(2)(b)               #  50x50x64
    u1 = layers.Concatenate()([u1, e2])          #  50x50x96  (skip)
    u1 = _conv_block(u1, 32, reg)                #  50x50x32
    u2 = layers.UpSampling2D(2)(u1)              # 100x100x32
    u2 = layers.Concatenate()([u2, e1])          # 100x100x48 (skip)
    u2 = _conv_block(u2, 16, reg)                # 100x100x16
    mask = layers.Conv2D(1, 1, padding="same", activation="sigmoid",
                         name="unet_mask")(u2)   # 100x100x1
    assert tuple(mask.shape[1:3]) == (IMG_SIZE, IMG_SIZE), \
        f"mask spatial dims {mask.shape[1:3]} != ({IMG_SIZE},{IMG_SIZE})"
    return mask

def build_model(name):
    """Build a frozen-backbone classifier. Call inside strategy.scope()."""
    reg = regularizers.l2(L2_REG)
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), name="input_0_255")

    x01      = layers.Rescaling(1.0 / 255.0, name="to_0_1")(inp)   # [0,1]
    mask     = micro_unet_attention(x01, reg)                      # (100,100,1)
    attended = layers.Multiply(name="apply_mask")([inp, mask])     # ~[0,255]

    if name == "EfficientNetB0":
        # EfficientNet normalizes internally -> feed [0,255] unchanged.
        adapted = layers.Rescaling(1.0, 0.0, name="adapter")(attended)
        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS), pooling=None)
    elif name == "MobileNetV2":
        # MobileNetV2 expects [-1, 1].
        adapted = layers.Rescaling(1.0 / 127.5, -1.0, name="adapter")(attended)
        backbone = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS), pooling=None)
    else:
        raise ValueError(f"unknown model: {name}")

    backbone.trainable = False                                     # FROZEN
    feats = backbone(adapted, training=False)
    g = layers.GlobalAveragePooling2D()(feats)
    g = layers.Dropout(DROPOUT)(g)
    g = layers.Dense(128, activation="relu", kernel_regularizer=reg)(g)
    g = layers.Dropout(DROPOUT)(g)
    out = layers.Dense(N_CLASSES, activation="softmax",
                       kernel_regularizer=reg, name="head")(g)
    return Model(inp, out, name=f"{name}_unetattn")

# sanity: confirm only U-Net + head are trainable
with strategy.scope():
    _dbg = build_model("MobileNetV2")
_tr = int(np.sum([np.prod(w.shape) for w in _dbg.trainable_weights]))
_nt = int(np.sum([np.prod(w.shape) for w in _dbg.non_trainable_weights]))
print(f"MobileNetV2  trainable={_tr:,}  frozen={_nt:,}")
del _dbg

/tmp/ipykernel_23/3144167661.py:55: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MobileNetV2  trainable=282,628  frozen=2,257,984


In [6]:
# ============================================================================
# CELL 6 - LOSO cross-validation with per-fold checkpointing & resume
# ----------------------------------------------------------------------------
# For each of the 13 sources: train on the other 12, predict the held-out
# source's patches, take a majority vote -> one source-level prediction.
# Models are crowned by SOURCE-LEVEL macro-F1. State is saved after every fold
# so a 12-hour timeout can be resumed (see notebook header).
# ============================================================================
def class_weights_for(df):
    """Inverse-frequency weights so the rare Osteoporosis class is not ignored."""
    counts = df["label"].value_counts().reindex(range(N_CLASSES)).fillna(0)
    total  = counts.sum()
    return {i: float(total / (N_CLASSES * c)) if c > 0 else 0.0
            for i, c in counts.items()}

def source_label(df, src):
    """The single ground-truth class of one source X-ray."""
    return int(df.loc[df["source"] == src, "label"].mode().iloc[0])

def _ckpt_path(model_name):
    return os.path.join(OUT_DIR, f"loso_ckpt_{model_name}.json")

def _find_ckpt(model_name):
    """Recursive search — works at any nesting depth Kaggle uses.
    Checks /kaggle/working first (same session), then all of /kaggle/input.
    This is why the previous runs kept re-running from scratch: the old code
    only searched one level deep and missed the nested checkpoint files."""
    fname = f"loso_ckpt_{model_name}.json"
    # 1. same session / current working dir
    local = os.path.join(OUT_DIR, fname)
    if os.path.exists(local):
        print(f"  [ckpt] found (working): {local}")
        return local
    # 2. any attached input dataset, searched recursively
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if matches:
        print(f"  [ckpt] found (input): {matches[0]}")
        return matches[0]
    print(f"  [ckpt] not found for {model_name} — starting fresh")
    return None

def _fresh_state():
    return {"completed": [], "src_true": [], "src_pred": [],
            "pooled_true": [], "pooled_pred": [], "per_fold": []}

def _load_state(model_name, sources):
    p = _find_ckpt(model_name)
    if p is None:
        return _fresh_state()
    with open(p) as f:
        state = json.load(f)
    # Guard: a checkpoint from a different dataset (sources don't match) must NOT
    # be trusted -- otherwise the parallel arrays misalign. Start fresh instead.
    valid = {str(s) for s in sources}
    stale = [s for s in state["completed"] if s not in valid]
    if stale:
        warnings.warn(f"{model_name}: checkpoint references sources {stale} absent "
                      f"from current data -> ignoring stale checkpoint, restarting.")
        return _fresh_state()
    print(f"  [ckpt] resuming {model_name}: {len(state['completed'])} folds done "
          f"{state['completed']}")
    return state

def _save_state(model_name, state):
    with open(_ckpt_path(model_name), "w") as f:
        json.dump(state, f)

def train_one_fold(model_name, tr_df):
    """Build a fresh frozen-backbone model and train its head to convergence."""
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)          # identical init every fold
    with strategy.scope():
        model = build_model(model_name)
        model.compile(optimizer=tf.keras.optimizers.Adam(INIT_LR),
                      loss="categorical_crossentropy", metrics=["accuracy"])
    callbacks = [
        # train to PEAK: stop only when training loss stops improving, then
        # restore the best weights. Never looks at the held-out source.
        tf.keras.callbacks.EarlyStopping(
            monitor="loss", patience=ES_PATIENCE,
            restore_best_weights=True, verbose=0),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="loss", factor=0.5, patience=LR_PATIENCE,
            min_lr=1e-6, verbose=0),
    ]
    hist = model.fit(make_dataset(tr_df, training=True),
                     epochs=EPOCHS_MAX, steps_per_epoch=steps_for(tr_df),
                     callbacks=callbacks,
                     class_weight=class_weights_for(tr_df), verbose=0)
    return model, len(hist.history["loss"])        # epochs actually run

def run_loso(model_name, manifest, sources):
    print(f"\n{'='*72}\nLOSO :: {model_name}\n{'='*72}")
    state = _load_state(model_name, sources)
    done  = set(state["completed"])
    t_start = time.time()

    for k, s in enumerate(sources, 1):
        if s in done:
            print(f"[fold {k:>2}/{len(sources)}] source={s}  SKIPPED (from checkpoint)")
            continue

        tr_df = manifest[manifest["source"] != s]
        va_df = manifest[manifest["source"] == s]
        assert len(tr_df) > 0 and len(va_df) > 0, f"empty fold for source {s}"
        truth = source_label(va_df, s)
        print(f"\n[fold {k:>2}/{len(sources)}] hold-out source={s}  "
              f"train={len(tr_df):,}  val={len(va_df):,}  true={CLASSES[truth]}")

        t0 = time.time()
        model, ran = train_one_fold(model_name, tr_df)

        probs  = model.predict(make_dataset(va_df, training=False), verbose=0)
        y_pred = probs.argmax(1)
        y_true = va_df["label"].to_numpy()
        n = min(len(y_pred), len(y_true))            # _keep may drop blanks
        y_pred, y_true = y_pred[:n], y_true[:n]
        assert len(y_pred) == len(y_true), "patch length mismatch"

        vote = int(np.bincount(y_pred, minlength=N_CLASSES).argmax())

        state["completed"].append(s)
        state["src_true"].append(truth)
        state["src_pred"].append(vote)
        state["pooled_true"].extend(y_true.tolist())
        state["pooled_pred"].extend(y_pred.tolist())
        state["per_fold"].append({"source": s, "true": CLASSES[truth],
                                  "pred": CLASSES[vote],
                                  "correct": int(vote == truth),
                                  "n_patches": int(n), "epochs": int(ran)})
        _save_state(model_name, state)               # persist every fold

        elapsed = time.time() - t_start
        eta = (elapsed / k) * (len(sources) - k) if k < len(sources) else 0
        print(f"   epochs={ran}  fold_time={time.time()-t0:.0f}s  "
              f"vote={CLASSES[vote]}  {'OK' if vote==truth else 'WRONG'}  "
              f"elapsed={elapsed/60:.1f}m  ETA~{eta/60:.1f}m  [ckpt saved]")

    assert len(state["src_true"]) == len(sources)
    f1  = f1_score(state["src_true"], state["src_pred"],
                   labels=range(N_CLASSES), average="macro", zero_division=0)
    acc = float(np.mean(np.array(state["src_true"]) == np.array(state["src_pred"])))
    wins = int(np.sum(np.array(state["src_true"]) == np.array(state["src_pred"])))
    print(f"\n{model_name} COMPLETE -> source macro-F1={f1:.4f}  "
          f"acc={acc:.4f} ({wins}/{len(sources)})")
    return {"model": model_name, **state, "source_macro_f1": f1,
            "source_accuracy": acc, "wins": wins, "n_sources": len(sources)}

results = {m: run_loso(m, manifest, sources) for m in MODELS}


LOSO :: EfficientNetB0
  [ckpt] found (input): /kaggle/input/datasets/manojkumar722004/osteo-checkpoints/loso_ckpt_EfficientNetB0.json
  [ckpt] resuming EfficientNetB0: 13 folds done ['roiant2', 'roiant4', 'roiant5', 'roiant6', 'roiant7', 'roiant9', 'roiant12', 'roiant14', 'roiant22', 'roiant26', 'roiant27', 'roiant28', 'roiant30']
[fold  1/13] source=roiant2  SKIPPED (from checkpoint)
[fold  2/13] source=roiant4  SKIPPED (from checkpoint)
[fold  3/13] source=roiant5  SKIPPED (from checkpoint)
[fold  4/13] source=roiant6  SKIPPED (from checkpoint)
[fold  5/13] source=roiant7  SKIPPED (from checkpoint)
[fold  6/13] source=roiant9  SKIPPED (from checkpoint)
[fold  7/13] source=roiant12  SKIPPED (from checkpoint)
[fold  8/13] source=roiant14  SKIPPED (from checkpoint)
[fold  9/13] source=roiant22  SKIPPED (from checkpoint)
[fold 10/13] source=roiant26  SKIPPED (from checkpoint)
[fold 11/13] source=roiant27  SKIPPED (from checkpoint)
[fold 12/13] source=roiant28  SKIPPED (from checkpoint)

/tmp/ipykernel_23/3144167661.py:55: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = tf.keras.applications.MobileNetV2(


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1781098500.547654      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1781098500.548120      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


...

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


   epochs=60  fold_time=2278s  vote=Osteoporosis  WRONG  elapsed=420.1m  ETA~35.0m  [ckpt saved]

[fold 13/13] hold-out source=roiant30  train=34,650  val=2,888  true=Osteopenia


/tmp/ipykernel_23/3144167661.py:55: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = tf.keras.applications.MobileNetV2(


INFO:tensorflow:Collective all_reduce tensors: 26 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   epochs=60  fold_time=2323s  vote=Normal  WRONG  elapsed=458.8m  ETA~0.0m  [ckpt saved]

MobileNetV2 COMPLETE -> source macro-F1=0.3956  acc=0.4615 (6/13)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [7]:
# ============================================================================
# CELL 7 - Reports, confusion matrices, and crown the winner
# ============================================================================
def write_report(res):
    st, sp = res["src_true"], res["src_pred"]
    lines = [
        f"LOSO REPORT :: {res['model']}",
        f"STRIDE={STRIDE}  EPOCHS_MAX={EPOCHS_MAX}  GLOBAL_BATCH={GLOBAL_BATCH}",
        "=" * 60,
        f"source-level accuracy : {res['source_accuracy']:.4f} "
            f"({res['wins']}/{res['n_sources']})",
        f"source-level macro-F1 : {res['source_macro_f1']:.4f}",
        f"chance (1/3)          : {1.0 / N_CLASSES:.4f}",
        "", "Per-class source-level precision / recall / F1:"]
    p, r, f, supp = precision_recall_fscore_support(
        st, sp, labels=range(N_CLASSES), zero_division=0)
    for i, c in enumerate(CLASSES):
        lines.append(f"  {c:<14} P={p[i]:.3f}  R={r[i]:.3f}  "
                     f"F1={f[i]:.3f}  support={int(supp[i])}")
    cm = confusion_matrix(st, sp, labels=range(N_CLASSES))
    lines += ["", "Source-level confusion matrix (rows=true, cols=pred):",
              "            " + "  ".join(f"{c[:6]:>6}" for c in CLASSES)]
    for i, c in enumerate(CLASSES):
        lines.append(f"  {c:<10}" + "  ".join(f"{v:>6d}" for v in cm[i]))
    lines += ["", "Patch-level classification report (reference only):",
              classification_report(res["pooled_true"], res["pooled_pred"],
                  labels=range(N_CLASSES), target_names=CLASSES, zero_division=0),
              "Per-fold detail:"]
    for d in res["per_fold"]:
        lines.append(f"  src={d['source']:<4} true={d['true']:<12} "
                     f"pred={d['pred']:<12} {'OK' if d['correct'] else 'X':<5} "
                     f"n={d['n_patches']:<5} epochs={d.get('epochs','?')}")
    text = "\n".join(lines)
    fp = os.path.join(OUT_DIR, f"cv_report_{res['model']}.txt")
    with open(fp, "w") as fh:
        fh.write(text)
    print(text); print(f"\n[saved] {fp}\n")

for m in MODELS:
    write_report(results[m])

# crown by SOURCE-LEVEL macro-F1 (tie-break: source accuracy)
winner = max(MODELS, key=lambda m: (results[m]["source_macro_f1"],
                                    results[m]["source_accuracy"]))
print("=" * 60, "\nFINAL COMPARISON")
for m in MODELS:
    r = results[m]
    print(f"  {m:<16} macro-F1={r['source_macro_f1']:.4f}  "
          f"acc={r['source_accuracy']:.4f}  ({r['wins']}/{r['n_sources']})")
print(f"\n>>> WINNER (source-level macro-F1): {winner}")

with open(os.path.join(OUT_DIR, "loso_summary.json"), "w") as fh:
    json.dump({m: {k: results[m][k] for k in
               ("source_macro_f1", "source_accuracy", "wins", "n_sources")}
               for m in MODELS} | {"winner": winner}, fh, indent=2)

LOSO REPORT :: EfficientNetB0
STRIDE=2  EPOCHS_MAX=60  GLOBAL_BATCH=128
source-level accuracy : 0.6154 (8/13)
source-level macro-F1 : 0.4786
chance (1/3)          : 0.3333

Per-class source-level precision / recall / F1:
  Normal         P=0.000  R=0.000  F1=0.000  support=3
  Osteopenia     P=0.714  R=0.833  F1=0.769  support=6
  Osteoporosis   P=0.600  R=0.750  F1=0.667  support=4

Source-level confusion matrix (rows=true, cols=pred):
            Normal  Osteop  Osteop
  Normal         0       2       1
  Osteopenia     0       5       1
  Osteoporosis     1       0       3

Patch-level classification report (reference only):
              precision    recall  f1-score   support

      Normal       0.12      0.08      0.09      8663
  Osteopenia       0.69      0.74      0.72     17325
Osteoporosis       0.54      0.63      0.58     11550

    accuracy                           0.55     37538
   macro avg       0.45      0.48      0.46     37538
weighted avg       0.51      0.55     

In [8]:
# ============================================================================
# CELL 8 - Retrain the winning model on ALL sources, then save for deployment
# ============================================================================
print(f"Retraining winner '{winner}' on all {len(manifest):,} patches ...")
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
with strategy.scope():
    final_model = build_model(winner)
    final_model.compile(optimizer=tf.keras.optimizers.Adam(INIT_LR),
                        loss="categorical_crossentropy", metrics=["accuracy"])

final_model.fit(
    make_dataset(manifest, training=True), epochs=EPOCHS_MAX,
    steps_per_epoch=steps_for(manifest),
    class_weight=class_weights_for(manifest),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="loss", patience=ES_PATIENCE,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5,
                                             patience=LR_PATIENCE, min_lr=1e-6,
                                             verbose=1)],
    verbose=2)

KERAS_PATH = os.path.join(OUT_DIR, f"osteo_{winner}_unetattn.keras")
H5_PATH    = os.path.join(OUT_DIR, "osteoporosis_mobilenetv2.h5")  # legacy name
final_model.save(KERAS_PATH)
try:
    final_model.save(H5_PATH)
except Exception as e:                               # pragma: no cover
    warnings.warn(f"H5 save failed (use the .keras file): {e}")
print(f"saved: {KERAS_PATH}")

Retraining winner 'EfficientNetB0' on all 37,538 patches ...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/60
INFO:tensorflow:Collective all_reduce tensors: 26 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1781126030.528778      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_unetattn_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


293/293 - 75s - 257ms/step - accuracy: 0.8962 - loss: 0.3166 - learning_rate: 0.0010
Epoch 2/60
293/293 - 57s - 194ms/step - accuracy: 0.9445 - loss: 0.1943 - learning_rate: 0.0010
Epoch 3/60
293/293 - 57s - 194ms/step - accuracy: 0.9569 - loss: 0.1506 - learning_rate: 0.0010
Epoch 4/60
293/293 - 57s - 193ms/step - accuracy: 0.9638 - loss: 0.1332 - learning_rate: 0.0010
Epoch 5/60
293/293 - 57s - 193ms/step - accuracy: 0.9699 - loss: 0.1192 - learning_rate: 0.0010
Epoch 6/60
293/293 - 57s - 193ms/step - accuracy: 0.9723 - loss: 0.1079 - learning_rate: 0.0010
Epoch 7/60
293/293 - 56s - 192ms/step - accuracy: 0.9766 - loss: 0.0972 - learning_rate: 0.0010
Epoch 8/60
293/293 - 56s - 192ms/step - accuracy: 0.9787 - loss: 0.0941 - learning_rate: 0.0010
Epoch 9/60
293/293 - 56s - 192ms/step - accuracy: 0.9787 - loss: 0.0910 - learning_rate: 0.0010
Epoch 10/60
293/293 - 56s - 192ms/step - accuracy: 0.9808 - loss: 0.0818 - learning_rate: 0.0010
Epoch 11/60
293/293 - 56s - 191ms/step - accuracy:

saved: /kaggle/working/osteo_EfficientNetB0_unetattn.keras


In [9]:
# ============================================================================
# CELL 9 - End-to-end inference: (YOLOv8 ROI) -> patches -> winning classifier
# ----------------------------------------------------------------------------
# The U-Net spatial attention is EMBEDDED in final_model, so it is applied
# automatically during predict(). A trained YOLOv8 mandible detector is optional:
# if no weights are supplied the code falls back to whole-image patching and
# warns -- it never fabricates an ROI.
# ============================================================================
import cv2

def _remove_border(img, thr=5):
    """Crop away uniform black/white margins around the X-ray."""
    _, th = cv2.threshold(img, thr, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(th)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    return img[y:y + h, x:x + w]

def _split_patches(gray, patch=IMG_SIZE):
    """Tile the (cropped) ROI into non-overlapping patch x patch tiles."""
    h, w = gray.shape[:2]
    return [gray[y:y + patch, x:x + patch]
            for y in range(0, h - patch + 1, patch)
            for x in range(0, w - patch + 1, patch)
            if gray[y:y + patch, x:x + patch].shape[:2] == (patch, patch)]

def _mandible_roi(bgr, yolo_weights):
    """Return (x1,y1,x2,y2). Whole image if no detector / no detection."""
    if not yolo_weights or not os.path.exists(yolo_weights):
        warnings.warn("No YOLOv8 weights -> using WHOLE image as ROI.")
        h, w = bgr.shape[:2]; return (0, 0, w, h)
    try:
        from ultralytics import YOLO
    except ImportError:
        warnings.warn("ultralytics not installed -> whole-image fallback.")
        h, w = bgr.shape[:2]; return (0, 0, w, h)
    res = YOLO(yolo_weights).predict(bgr, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        warnings.warn("YOLOv8 found no ROI -> whole-image fallback.")
        h, w = bgr.shape[:2]; return (0, 0, w, h)
    confs = res.boxes.conf.cpu().numpy()
    xyxy  = res.boxes.xyxy.cpu().numpy()[int(confs.argmax())]
    return tuple(int(round(v)) for v in xyxy)

def predict_panoramic(image_path, model=None, yolo_weights=None,
                      return_details=False):
    """Full pipeline for one panoramic X-ray -> majority-vote diagnosis."""
    if model is None:
        model = final_model
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(image_path)
    x1, y1, x2, y2 = _mandible_roi(bgr, yolo_weights)
    gray = _remove_border(cv2.cvtColor(bgr[max(0, y1):y2, max(0, x1):x2],
                                       cv2.COLOR_BGR2GRAY))
    patches = _split_patches(gray)
    if not patches:
        raise ValueError("ROI too small to yield any 100x100 patch.")
    kept = [p for p in patches
            if MEAN_LO <= p.mean() <= MEAN_HI and p.std() > STD_MIN] or patches
    batch = np.stack([cv2.cvtColor(p, cv2.COLOR_GRAY2RGB)
                      for p in kept]).astype("float32")        # [0,255]
    probs = model.predict(batch, verbose=0)
    votes = probs.argmax(1)
    assert len(votes) == len(batch), "inference length mismatch"
    idx = int(np.bincount(votes, minlength=N_CLASSES).argmax())
    if return_details:
        return {"diagnosis": CLASSES[idx],
                "confidence": float(probs.mean(0)[idx]),
                "n_patches": len(batch), "roi_xyxy": (x1, y1, x2, y2),
                "vote_counts": dict(zip(CLASSES,
                    np.bincount(votes, minlength=N_CLASSES).tolist()))}
    return CLASSES[idx]

# Example (uncomment and set a real path):
# print(predict_panoramic("/kaggle/input/.../full_panoramic.jpg",
#                         yolo_weights="/kaggle/input/.../mandible_yolov8.pt",
#                         return_details=True))
print("predict_panoramic() ready.")

predict_panoramic() ready.


In [10]:
# ============================================================================
# CELL 10 - Brightness baseline (honesty anchor) under the SAME LOSO protocol
# ----------------------------------------------------------------------------
# A trivial mean-intensity logistic model. If the frozen-backbone CNNs cannot
# beat this, that is an honest, reportable finding about the architecture.
# ============================================================================
from sklearn.linear_model import LogisticRegression

def _patch_mean(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return float(g.mean()) if g is not None else 0.0

mf = manifest.copy()
mf["feat"] = mf["path"].map(_patch_mean)

b_true, b_pred = [], []
for s in sources:
    tr = mf[mf["source"] != s]
    va = mf[mf["source"] == s]
    clf = LogisticRegression(max_iter=1000, class_weight="balanced",
                             random_state=SEED)
    clf.fit(tr[["feat"]].to_numpy(), tr["label"].to_numpy())
    vote = int(np.bincount(clf.predict(va[["feat"]].to_numpy()),
                           minlength=N_CLASSES).argmax())
    b_pred.append(vote); b_true.append(source_label(va, s))

b_f1  = f1_score(b_true, b_pred, labels=range(N_CLASSES),
                 average="macro", zero_division=0)
b_acc = float(np.mean(np.array(b_true) == np.array(b_pred)))
b_win = int(sum(t == p for t, p in zip(b_true, b_pred)))
print(f"Brightness baseline -> source macro-F1={b_f1:.4f}  "
      f"acc={b_acc:.4f} ({b_win}/{len(sources)})")
with open(os.path.join(OUT_DIR, "cv_report_BrightnessBaseline.txt"), "w") as fh:
    fh.write(f"Brightness baseline (LOSO)\nmacro-F1={b_f1:.4f}  acc={b_acc:.4f} "
             f"({b_win}/{len(sources)})\n\n")
    fh.write(classification_report(b_true, b_pred, labels=range(N_CLASSES),
                                   target_names=CLASSES, zero_division=0))
print("saved cv_report_BrightnessBaseline.txt")

Brightness baseline -> source macro-F1=0.5333  acc=0.5385 (7/13)
saved cv_report_BrightnessBaseline.txt
